# DISTILBERT TRAINING

In [ ]:
import pandas as pd

# Monte Google Drive pour accéder aux fichiers
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## Data Loading

In [ ]:
# Charger le dataset depuis Google Drive
df = pd.read_csv('/content/drive/MyDrive/FAKE_NEWS_PROJECT/full_dataset.csv')
df.head()

,Unnamed: 0,text,label
0,18549,"DHAKA, (Reuters) - Bangladesh and Myanmar agr...",1
1,1133,18 percent of our land in our state right now ...,1
2,21643,Hahahahahahahaha. Hahahahahahahahahahahahahaha...,0
3,32595,"Omarosa Manigault, a senior staff member of Pr...",0
4,12687,"The port provides more than 297,000 jobs direc...",1


## Data Exploration and Preprocessing

In [ ]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Labels: {df["label"].value_counts()}")

Shape: (46687, 3)
Columns: ['Unnamed: 0', 'text', 'label']
Labels: label
1    25694
0    20993
Name: count, dtype: int64


## GPU Setup

In [ ]:
import torch
# Verification du GPU
print(f"GPU disponible: {torch.cuda.is_available()}")
# Nom du GPU
print(f"GPU: {torch.cuda.get_device_name(0)}")
# VRAM
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

GPU disponible: True
GPU: Tesla T4
VRAM: 15.6 Go


## Tokenization

In [ ]:
import transformers as tf

tokenizer = tf.AutoTokenizer.from_pretrained("distilbert-base-uncased")
ex = tokenizer("Hello World")
print(ex)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


In [ ]:
print(tokenizer.decode([101]))
print(tokenizer.decode([102]))

[CLS]
[SEP]


"Hello World" c'est 2 mots mais on a 4 input_ids :
- 101 = [CLS] — marqueur de début de séquence.
- 102 = [SEP] — marqueur de fin de séquence.

### Tokenization example with padding - LIAR (short text)

In [ ]:
# Premier text (text court)
example = df["text"].iloc[0]

In [ ]:
tokens = tokenizer(example, max_length=512, truncation=True, padding="max_length")

print(f"Example: {example[:100]}...")
print(f"Tokens numbers: {sum(tokens["attention_mask"])}")
print(f"Total length (avec padding): {len(tokens["input_ids"])}")

Example: DHAKA,  (Reuters) - Bangladesh and Myanmar agreed on Monday to set up a  joint working group  on the...
Tokens numbers: 119
Total length (avec padding): 512


Le texte fait 119 vrais tokens. Mais avec max_length=512, le tokenizer a rajouté 393 tokens de padding (des zéros) pour atteindre 512. C'est à ça que sert l'attention_mask. Il informe le modèle : "les 119 premiers sont du vrai texte, le reste c'est du remplissage à ignorer."

### Tokenization example without padding - ISOT (long text)

In [ ]:
# Récupère un text long
longest_id = df["text"].str.len().idxmax()
example = df["text"].iloc[longest_id]

In [ ]:
tokens = tokenizer(example, max_length=512, truncation=True, padding="max_length")

print(f"Example: {example[:100]}...")
print(f"Nb tokens: {sum(tokens["attention_mask"])}")
print(f"Longueur totale (avec padding): {len(tokens["input_ids"])}")

Example:  Funny how secrets travel. I d start to believe, if I were to bleed.    Lyrics written by David Bowi...
Nb tokens: 512
Longueur totale (avec padding): 512


512 tokens, zéro padding, ce texte a été tronqué. Tout ce qui dépasse 512 tokens est perdu.

## Data Splitting

Pour la baseline on a utlisé 80/20 (train/test). Pour DistilBERT, on a besoin d'un validation set afin gerer la perte pendant l'entraînement et éviter l'overfitting.

**Répartition classique : 80% train / 10% validation / 10% test**

In [ ]:
from sklearn.model_selection import train_test_split

# Découpage -> entrainement (80%) & données de test temporaires (20%)
# 'stratify=df["label"]' asssure que la proportion de label est maintenue entre chaque sets
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], train_size = 0.8, random_state = 42, shuffle = True, stratify = df["label"])

# Découpage restant des données de test temporaires en validation (10%) et test (10%)
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, train_size=0.5, random_state = 42 , shuffle = True, stratify = y_test)

print(f"Train: {len(X_train)} / Valid: {len(X_valid)} / Test: {len(X_test)}")
print(f"Train labels:\n{y_train.value_counts()}")

Train: 37349 / Valid: 4669 / Test: 4669
Train labels:
label
1    20555
0    16794
Name: count, dtype: int64


## Tokenisation du dataset

Le Trainer de Hugging Face ne travaille pas avec des DataFrames pandas. Il attend un objet Dataset de la librairie datasets. C'est un format optimisé pour le traitement par batch, le mapping de fonctions (comme la tokenisation), et le chargement en mémoire efficace.

In [ ]:
from datasets import Dataset

# Création des datasets au format Hugging Face
train_dataset = Dataset.from_dict({"text": X_train.to_list(), "label": y_train.to_list()})
valid_dataset = Dataset.from_dict({"text": X_valid.to_list(), "label": y_valid.to_list()})
test_dataset = Dataset.from_dict({"text": X_test.to_list(), "label": y_test.to_list()})

# Fonction de tokenisation
def tokenize(batch):
  return tokenizer(batch["text"], max_length=512, truncation=True, padding="max_length")

# On Tokenize les 3 datasets
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/37349 [00:00<?, ? examples/s]

Map:   0%|          | 0/4669 [00:00<?, ? examples/s]

Map:   0%|          | 0/4669 [00:00<?, ? examples/s]

In [ ]:
print(train_dataset)
print(train_dataset[0].keys())

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 37349
})
dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


Le dataset contient maintenant les **5 colonnes** : text et label d'origine, plus input_ids, attention_mask et token_type_ids ajoutés par le tokenizer.

Le Trainer n'a besoin que de input_ids, attention_mask et label. La colonne text ne sert plus (DistilBERT travaille avec les tokens, pas le texte brut) et token_type_ids est inutile pour DistilBERT.

## Pre-trained model loading

DistilBert a été créé "deviner" un mot manquant dans une phrase.

Pour cela, il y a deux parties:
- le cerveau -> Les 6 couches d'attention qui comprennent le language. C'est grâce à ces 6 couches d'attention que le modèle apprend la grammaire, le sens des mots et les relations contextuelles.
- la tête "deviner les mots" -> C'est la couche à la fin qui prend la compréhension du cerveau et produit un mot.

Ici, en appliquant *AutoModelForSequenceClassification*, on garde le cerveau du modèle pré-entrainé et l'on se débarasse de la tête "deviner les mots" pour la remplacer par notre nouvelle tête "classifier en 2 classes (fake/real).

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) # num_labels = 2 -> 2 classes (fake/real)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


On peut observer ce qu'il s'est passé. On voit que l'ancienne tête du modèle (status *UNEXPECTED*) a été remplacée par la nouvelle tête de classification (status *MISSING*).

UNEXPECTED -> se débarasse des composants de l'ancienne tête.
MISSING -> Ajoute les composants nécéssaire à la nouvelle tête de classification.

## Fine Tuning

In [ ]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import f1_score, classification_report

def get_metrics(eval_prediction):
  logits, labels = eval_prediction # logits -> score brut en sortie du model / labels -> représente les vrais labels pour comparer aux scores bruts
  preds = np.argmax(logits, axis=-1) # on récupère l'index du score le plus élevé
  f1 = f1_score(labels, preds, average="weighted") # calcul du score F1
  return {"f1": f1} # retourne le score F1

# Parametre d'entrainement - Avec les hyperparamètres standard de BERT
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3, # Le nombre de fois que le model voit le dataset passer
    per_device_train_batch_size=16, # le nombre de text a traiter en même temps
    per_device_eval_batch_size=16, # idem mais pour le dataset d'évaliation
    learning_rate=2e-5, # Vitesse d'apprentissage -> Trop hau: le modele "oublie", / Trop bas : il n'apprend pas
    weight_decay=0.01, # Régularisation pour éviter l'overfitting : C'est l'équivalent du paramètre C dans logistic regression
    eval_strategy="epoch", # Evalue le modèle sur le set de validation après epoch
    save_strategy="epoch", # sauvegarde un checkpoint à la fin de chaque epoch
    load_best_model_at_end=True, # Garde le meilleur score (ici F1) des trois dernier epoch
    metric_for_best_model="f1", # le scrore qui determoinbe le meilleur modele
    fp16=True, # Réduit la VRAM utilisé (travail en 16 bits au lieu de 32 bits), accélère l'entrainement
    report_to="none", # Pas de logging
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    compute_metrics = get_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.118241,0.110267,0.945626
2,0.097477,0.126215,0.949190
3,0.064628,0.177436,0.946215


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=7005, training_loss=0.09582518911804155, metrics={'train_runtime': 1559.698, 'train_samples_per_second': 71.839, 'train_steps_per_second': 4.491, 'total_flos': 1.4842574617208832e+16, 'train_loss': 0.09582518911804155, 'epoch': 3.0})

On observe entre chaque epoch une chute du *Training Loss* et cela est normal puisque le modèle apprend de mieux en mieux sur les données d'entrainement.

Cependant, la *Validation Loss* augmente entre l'epoch 2 et l'epoch 3. Cela signifie que le modèle devient meilleur sur les données qu'il connait, mais moins bon sur les données qu'il n'a jamais vues.

**C'est un signal d'OVERFITTING** -> Le modèle commence à mémoriser les exemples d'entraînement au lieu d'apprendre des règles générales.

C'est tout l'intérêt du paramètres *load_best_model_at_end=True*, le Trainer regarde quel epoch avait le meilleur F1 sur la validation et recharge celui ci. Sans ce paramètre, le modèle de l'epoch final aurait été gardé.

## Evaluation

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(y_test, preds, target_names=["Fake (0)", "Real (1)"]))

              precision    recall  f1-score   support

    Fake (0)       0.95      0.93      0.94      2100
    Real (1)       0.94      0.96      0.95      2569

    accuracy                           0.95      4669
   macro avg       0.95      0.95      0.95      4669
weighted avg       0.95      0.95      0.95      4669



## Comparaison avec le baseline

| Modèle | F1 | Precision | Recall |
| --- | --- | --- | --- |
| TF-IDF + Logistic Regression | 0.90 | 0.90 | 0.90 |
| DistilBERT fine-tuné | **0.95** | **0.95** | **0.95** |

Amélioration de 5 points de F1. La compréhension contextuelle de DistilBERT apporte une vraie valeur par rapport à l'approche par fréquence de mots.

In [ ]:
test_texts = [
    "The government has announced a new policy to combat climate change.",
    "The spokesman for the ministry issued a statement saying the government would seek to address the allegations raised by the parliamentary committee investigating the matter.",
    "You won't believe what this racist politician just revealed about his secret scheme. Watch the video and share this story before they try to hide the truth from liberal America."
]

from transformers import pipeline
clf = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)
for text in test_texts:
    result = clf(text)
    print(f"{text[:80]}...")
    print(f"  -> {result}\n")

The government has announced a new policy to combat climate change....
  → [{'label': 'LABEL_0', 'score': 0.942294716835022}]

The spokesman for the ministry issued a statement saying the government would se...
  → [{'label': 'LABEL_0', 'score': 0.7845667600631714}]

You won't believe what this racist politician just revealed about his secret sch...
  → [{'label': 'LABEL_0', 'score': 0.9989412426948547}]



In [ ]:
save_path = "/content/drive/MyDrive/FAKE_NEWS_PROJECT/distilbert_model"
model.config.id2label = {0: "Fake", 1: "Real"}
model.config.label2id = {"Fake": 0, "Real": 1}
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Modèle sauvegardé dans {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle sauvegardé dans /content/drive/MyDrive/FAKE_NEWS_PROJECT/distilbert_model
